In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
# load_dotenv(dotenv_path=Path(".env"))
print(os.getenv("GROQ_API_KEY"))

In [2]:
import json

# Point directly to the single JSON file
file_path = './documents/currencies.json'

with open(file_path, 'r', encoding='utf-8') as f:
    documents = json.load(f)

documents

{'currencies': [{'code': 'USD',
   'name': 'US Dollar',
   'symbol': '$',
   'decimal_digits': 2,
   'countries': ['United States', 'Ecuador', 'El Salvador']},
  {'code': 'EUR',
   'name': 'Euro',
   'symbol': '€',
   'decimal_digits': 2,
   'countries': ['Germany', 'France', 'Italy', 'Spain', 'Netherlands']},
  {'code': 'GBP',
   'name': 'British Pound',
   'symbol': '£',
   'decimal_digits': 2,
   'countries': ['United Kingdom']},
  {'code': 'JPY',
   'name': 'Japanese Yen',
   'symbol': '¥',
   'decimal_digits': 0,
   'countries': ['Japan']},
  {'code': 'CHF',
   'name': 'Swiss Franc',
   'symbol': 'CHF',
   'decimal_digits': 2,
   'countries': ['Switzerland', 'Liechtenstein']},
  {'code': 'AUD',
   'name': 'Australian Dollar',
   'symbol': 'A$',
   'decimal_digits': 2,
   'countries': ['Australia']},
  {'code': 'CAD',
   'name': 'Canadian Dollar',
   'symbol': 'C$',
   'decimal_digits': 2,
   'countries': ['Canada']},
  {'code': 'CNY',
   'name': 'Chinese Yuan',
   'symbol': '¥',
 

In [3]:
from ingest import load_currency_data
documents = load_currency_data()

In [14]:
import json
from minsearch import Index


def load_currency_data():
    file_path = "./documents/currencies.json"

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = data["currencies"]

    for doc in documents:
        # Use currency code as the unique document ID
        doc["doc_id"] = doc["code"]

    return documents



def build_index(documents):
    # Convert countries lists into searchable strings
    for doc in documents:
        if isinstance(doc.get("countries"), list):
            doc["countries"] = " ".join(doc["countries"])

    index = Index(
        text_fields=["code", "name", "symbol", "countries"],
        keyword_fields=["decimal_digits"]
    )

    index.fit(documents)

    return index


def search_currencies(index, query, num_results=5):
    results = index.search(
        query,
        boost_dict={
            "code": 3,
            "name": 3,
            "symbol": 2,
            "countries": 1
        },
        num_results=num_results
    )

    return results


if __name__ == "__main__":
    documents = load_currency_data()
    index = build_index(documents)

    results = search_currencies(index, "Japanese Yen")

    for result in results:
        print(result)


{'code': 'JPY', 'name': 'Japanese Yen', 'symbol': '¥', 'decimal_digits': 0, 'countries': 'Japan', 'doc_id': 'JPY'}


In [4]:
from ingest import load_currency_data, build_index

documents = load_currency_data()
index = build_index(documents)

In [5]:
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [6]:
def llm(prompt):
    response = openai_client.responses.create(
        # model = 'gpt-5.4-mini',
        model="openai/gpt-oss-120b",
        input=prompt,
    )

    return response.output_text

In [7]:
question = 'What currencies are supported?'
answer = llm(question)
print(answer)

The list of currencies a service can accept varies from one platform to another, but most modern payment‑and‑fintech solutions support the major world currencies. Below is a typical “core” set that you’ll find on most platforms, plus a note about extended coverage.

---

## Commonly Supported Currencies (Core Set)

| Currency | ISO‑4217 Code | Typical Symbol |
|----------|--------------|----------------|
| United States Dollar | **USD** | $ |
| Euro | **EUR** | € |
| British Pound Sterling | **GBP** | £ |
| Canadian Dollar | **CAD** | C$ |
| Australian Dollar | **AUD** | A$ |
| Japanese Yen | **JPY** | ¥ |
| Swiss Franc | **CHF** | Fr. |
| New Zealand Dollar | **NZD** | NZ$ |
| Singapore Dollar | **SGD** | S$ |
| Hong Kong Dollar | **HKD** | HK$ |
| Swedish Krona | **SEK** | kr |
| Norwegian Krone | **NOK** | kr |
| Danish Krone | **DKK** | kr |
| South Korean Won | **KRW** | ₩ |
| Chinese Yuan (Renminbi) | **CNY** | ¥ |
| Indian Rupee | **INR** | ₹ |
| Brazilian Real | **BRL** | R$ |


In [8]:
question = 'What currency is used in South Korea?'
answer = llm(question)
print(answer)

The currency used in South Korea is the **South Korean won**, abbreviated **KRW** and symbolized as **₩**.


In [14]:
%pip install rag_helper

  Using cached rag_helper-0.0.8-py3-none-any.whl.metadata (2.2 kB)
  Using cached rag_helper-0.0.6-py3-none-any.whl.metadata (2.2 kB)
  Using cached rag_helper-0.0.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached rag_helper-0.0.8-py3-none-any.whl.metadata (2.2 kB)
  Using cached rag_helper-0.0.6-py3-none-any.whl.metadata (2.2 kB)
  Using cached google_generativeai-0.7.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached instructor-1.17.0-py3-none-any.whl.metadata (13 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of rag-helper to determine which version is compatible with other requirements. This could take a while.
  Using cached rag_helper-0.0.1-py3-none-any.whl.metadata (626 bytes)
  Using cached rag_helper-0.0.0-py3-none-any.whl.metadata (666 bytes)

The conflict is caused by:
    rag-helper 0.0.4 depends on lancedb<0.9.0 and >=0.8.2
    rag-helper 0.0.1 depends on lancedb<0.9.0 and >=0.8.2
    rag-helper 0.0.0 depend

Requested rag_helper from https://files.pythonhosted.org/packages/30/4a/4b7f832e8a61296ef40df8d25c3a7cce98e8ead340b0912c17eb4680ba26/rag_helper-0.0.8-py3-none-any.whl has invalid metadata: Local version label can only be used with `==` or `!=` operators
    torch (>=2.3.1+cpu,<3.0.0)
           ~~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested rag_helper from https://files.pythonhosted.org/packages/6d/2e/fb213949a4d445ad8f55ad67b11bf78cf53950060d2fa359e3178bef3df5/rag_helper-0.0.6-py3-none-any.whl has invalid metadata: Local version label can only be used with `==` or `!=` operators
    torch (>=2.3.1+cpu,<3.0.0)
           ~~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested rag_helper from https://files.pythonhosted.org/packages/30/4a/4b7f832e8a61296ef40df8d25c3a7cce98e8ead340b0912c17eb4680ba26/rag_helper-0.0.8-py3-none-any.whl has invalid metadata: Local version label can only be used with `==` or `!=` operators
    torch (>=2.3.1+cpu,<3.0.0)

In [20]:
%pip install RAGBase

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
documents = load_currency_data()
index = build_index(documents)

assistant = RAGBase(index, openai_client)

In [10]:
from rag_helper import RAGBase
from ingest import load_currency_data, build_index

In [21]:
%pip install rag

INFO: pip is looking at multiple versions of rag to determine which version is compatible with other requirements. This could take a while.
  Using cached celery-5.0.5-py3-none-any.whl.metadata (19 kB)
  Using cached celery-5.0.4-py3-none-any.whl.metadata (19 kB)
  Using cached celery-5.0.3-py3-none-any.whl.metadata (19 kB)
  Using cached celery-5.0.2-py3-none-any.whl.metadata (19 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached py-1.11.0-py2.py3-none-any.whl.metadata (2.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  

Requested celery<5.1.0,>=5.0.2 from https://files.pythonhosted.org/packages/95/79/d73247b42082076bb32c828e3361cda2604ee523aa3bfe972bdfabffc6d0/celery-5.0.5-py3-none-any.whl (from rag) has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    pytz (>dev)
         ~^
Please use pip<24.1 if you need to use this version.
Requested celery<5.1.0,>=5.0.2 from https://files.pythonhosted.org/packages/c6/ff/a3ee60907625086fa5cac951811bd97d98b146f1ad557370f85ef3aaa007/celery-5.0.4-py3-none-any.whl (from rag) has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    pytz (>dev)
         ~^
Please use pip<24.1 if you need to use this version.
Requested celery<5.1.0,>=5.0.2 from https://files.pythonhosted.org/packages/74/37/3a88558d94f665e98abc9ab6cb1deeabd37ace7e287a650715a0d177ae6d/celery-5.0.3-py3-none-any.whl (from rag) has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS

In [12]:
assistant.rag('How do I submit homework?')

"I don't know."

In [13]:
assistant.rag('What currency is used in South Korea')

'South Korea uses the South Korean Won (currency code\u202fKRW, symbol\u202f₩).'

In [35]:
def rag(self, question):

        search_results = self.search(question)

        if not search_results:
            return "I don't know."

        prompt = self.build_prompt(
            question,
            search_results
        )

        return self.llm(prompt)

In [39]:
%pip install sqlitesearch

  Using cached sqlitesearch-0.3.0-py3-none-any.whl.metadata (15 kB)
Using cached sqlitesearch-0.3.0-py3-none-any.whl (55 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------------------------------------- -- 2.6/2.8 MB 20.6 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 16.7 MB/s  0:00:00
   ---------------------------------------- 0.0/43.0 MB ? eta -:--:--
   ----- ---------------------------------- 5.8/43.0 MB 27.6 MB/s eta 0:00:02
   ---------- ----------------------------- 11.5/43.0 MB 27.4 MB/s eta 0:00:02
   ------------ --------------------------- 13.9/43.0 MB 23.3 MB/s eta 0:00:02
   -------------- ------------------------- 15.7/43.0 MB 18.9 MB/s eta 0:00:02
   ---------------- ----------------------- 17.3/43.0 MB 16.0 MB/s eta 0:00:02
   ----------------- ---------------------- 18.9/43.0 MB 14.6 MB/s eta 0:00:02
   ------------------ --------------------- 19.4/43.0 MB 13.1 MB/s eta 0:00:02
   ------------------ -----------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from sqlitesearch import TextSearchIndex

In [15]:
index

In [16]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [25]:
def search(question, country=None):
    boost_dict = {
        'code': 3.0,
        'name': 3.0,
        'symbol': 2.0,
        'countries': 2.0
    }

    query = question

    if country:
        query = f"{question} {country}"

    return index.search(
        query,
        boost_dict=boost_dict,
        num_results=5
    )

In [27]:
search_results = search(question, 'United States')
search_results

[{'code': 'KRW',
  'name': 'South Korean Won',
  'symbol': '₩',
  'decimal_digits': 0,
  'countries': 'South Korea',
  'doc_id': 'KRW'},
 {'code': 'ZAR',
  'name': 'South African Rand',
  'symbol': 'R',
  'decimal_digits': 2,
  'countries': 'South Africa',
  'doc_id': 'ZAR'},
 {'code': 'USD',
  'name': 'US Dollar',
  'symbol': '$',
  'decimal_digits': 2,
  'countries': 'United States Ecuador El Salvador',
  'doc_id': 'USD'},
 {'code': 'GBP',
  'name': 'British Pound',
  'symbol': '£',
  'decimal_digits': 2,
  'countries': 'United Kingdom',
  'doc_id': 'GBP'},
 {'code': 'AED',
  'name': 'UAE Dirham',
  'symbol': 'د.إ',
  'decimal_digits': 2,
  'countries': 'United Arab Emirates',
  'doc_id': 'AED'}]

In [26]:
search_results = search(question, 'South Africa')
search_results

[{'code': 'KRW',
  'name': 'South Korean Won',
  'symbol': '₩',
  'decimal_digits': 0,
  'countries': 'South Korea',
  'doc_id': 'KRW'},
 {'code': 'ZAR',
  'name': 'South African Rand',
  'symbol': 'R',
  'decimal_digits': 2,
  'countries': 'South Africa',
  'doc_id': 'ZAR'}]

In [28]:
search_results = search(question, 'China')
search_results

[{'code': 'KRW',
  'name': 'South Korean Won',
  'symbol': '₩',
  'decimal_digits': 0,
  'countries': 'South Korea',
  'doc_id': 'KRW'},
 {'code': 'ZAR',
  'name': 'South African Rand',
  'symbol': 'R',
  'decimal_digits': 2,
  'countries': 'South Africa',
  'doc_id': 'ZAR'},
 {'code': 'CNY',
  'name': 'Chinese Yuan',
  'symbol': '¥',
  'decimal_digits': 2,
  'countries': 'China',
  'doc_id': 'CNY'}]

In [35]:

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(f"Currency code: {doc['code']}")
        lines.append(f"Currency name: {doc['name']}")
        lines.append(f"Symbol: {doc['symbol']}")
        lines.append(f"Decimal digits: {doc['decimal_digits']}")
        lines.append(f"Countries: {doc['countries']}")
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)

    prompt = f"""
QUESTION: {question}

CONTEXT:
{context}
""".strip()

    return prompt

In [36]:
prompt = build_prompt(question, search_results)
print(prompt)

QUESTION: What currency is used in South Korea?

CONTEXT:
Currency code: KRW
Currency name: South Korean Won
Symbol: ₩
Decimal digits: 0
Countries: South Korea

Currency code: ZAR
Currency name: South African Rand
Symbol: R
Decimal digits: 2
Countries: South Africa

Currency code: CNY
Currency name: Chinese Yuan
Symbol: ¥
Decimal digits: 2
Countries: China
